# Pillar 3 — Big Data Explorer

Explores the outputs of `spark_pipeline.py` (5 Parquet tables) and `kafka_streaming.py` (Kafka run summary), right here in VS Code. Reads the Spark output with pyarrow (using the Windows `\?\` long-path prefix — the partitioned Parquet paths under this folder exceed 260 characters, which DuckDB's `read_parquet()` can't open) and queries it with DuckDB.

Kernel: any **Python 3** kernel with duckdb/pandas/pyarrow — select it top-right if VS Code opens with a different one.

Run `spark_pipeline.py` and `kafka_streaming.py` first if the paths below don't exist yet (see `HOW_TO_RUN.md`).

In [1]:
import json
import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

# Parquet is a gitignored build artifact (outputs/artifacts/), separate from
# the tracked deliverables in outputs/results/ — see spark_pipeline.py.
SPARK_DIR = "../../../../../outputs/artifacts/anil_vaikuntam/03_big_data/spark"
KAFKA_SUMMARY = "../../../../../outputs/results/anil_vaikuntam/03_big_data/kafka/summary.json"

con = duckdb.connect(":memory:")


def read_table(name, partitioned=True):
    # Windows long-path prefix: these Parquet paths exceed 260 characters.
    import os
    import pyarrow.dataset as ds
    root = os.path.abspath(os.path.join(SPARK_DIR, name))
    if os.name == "nt":
        root = "\\\\?\\" + root
    arrow_tbl = ds.dataset(root, format="parquet", partitioning="hive" if partitioned else None).to_table()
    return con.sql("SELECT * FROM arrow_tbl").df()

## project_activity_summary

In [2]:
read_table("project_activity_summary").head(10)

,project_id,total_events,escalation_count,task_completions,document_uploads,last_event_timestamp,unique_users,unique_event_types
0,PRJ0296,146,4,24,17,2025-12-25 13:56:18,138,13
1,PRJ0372,145,2,28,17,2025-12-30 05:57:06,129,11
2,PRJ0381,144,6,26,13,2025-12-29 13:08:40,128,12
3,PRJ0285,144,2,23,19,2025-12-31 02:13:23,131,12
4,PRJ0390,141,4,22,20,2025-12-30 13:38:13,128,12
5,PRJ0266,136,5,22,14,2025-12-27 20:49:34,131,13
6,PRJ0128,136,5,31,20,2025-12-29 23:40:10,122,11
7,PRJ0175,134,7,34,10,2025-12-31 18:41:57,124,13
8,PRJ0052,133,8,20,7,2025-12-29 18:37:17,127,12
9,PRJ0029,133,5,23,14,2025-12-30 00:45:43,123,12


## user_activity_summary

In [3]:
read_table("user_activity_summary").head(10)

,user_id,login_count,logout_count,actions_taken,projects_touched,first_active,last_active,active_days
0,EMP0806,28,21,84,78,2025-01-03 12:04:45,2025-12-26 23:23:32,102
1,EMP0601,27,12,82,75,2025-01-02 11:58:08,2025-12-27 14:14:58,100
2,EMP0895,29,23,78,67,2025-01-02 09:12:52,2025-12-20 17:05:53,110
3,EMP0193,29,17,77,72,2025-01-10 04:10:38,2025-12-31 01:49:44,103
4,EMP0268,24,25,76,73,2025-01-02 18:14:45,2025-12-31 23:42:11,107
5,EMP0737,30,24,76,69,2025-01-01 05:16:27,2025-12-28 19:59:39,110
6,EMP0765,23,33,76,70,2025-01-02 01:25:11,2025-12-30 10:44:12,116
7,EMP0729,28,23,76,68,2025-01-07 00:07:02,2025-12-29 02:13:00,111
8,EMP0220,29,23,75,71,2025-01-02 05:15:27,2025-12-30 03:30:05,101
9,EMP0266,27,18,75,72,2025-01-04 18:40:54,2025-12-30 17:46:14,107


## escalation_log — resolved vs. unresolved, resolution time sanity check

In [4]:
esc = read_table("escalation_log")
print(esc["resolved"].value_counts())
print("\nnegative resolution_time_hours (should be 0):", (esc.loc[esc["resolved"], "resolution_time_hours"] < 0).sum())
esc[esc["resolved"]].sort_values("resolution_time_hours").head(10)

resolved
True     1098
False     871
Name: count, dtype: int64

negative resolution_time_hours (should be 0): 0


,event_id,project_id,raised_by,raised_at,resolved,resolved_by,resolved_at,resolution_time_hours,severity
1350,EVT0092244,PRJ0205,EMP0852,2025-12-16 17:01:33,True,EMP0924,2025-12-17 03:01:02,9.991389,Medium
1787,EVT0014198,PRJ0252,EMP0426,2025-02-25 06:32:09,True,EMP0140,2025-02-25 17:58:54,11.445833,Medium
1161,EVT0001476,PRJ0485,EMP0387,2025-01-17 19:21:46,True,EMP0677,2025-01-18 07:11:32,11.829444,Low
873,EVT0073988,PRJ0255,EMP0366,2025-09-17 01:38:29,True,EMP0741,2025-09-17 15:32:17,13.896667,Low
788,EVT0010840,PRJ0021,EMP0056,2025-02-27 12:38:27,True,EMP0840,2025-02-28 03:53:01,15.242778,Low
124,EVT0041174,PRJ0037,EMP0527,2025-05-20 14:43:58,True,EMP0580,2025-05-21 07:09:59,16.433611,Critical
1868,EVT0018393,PRJ0305,EMP0813,2025-03-09 19:48:39,True,EMP0388,2025-03-10 12:48:12,16.992500,Medium
1815,EVT0087193,PRJ0390,EMP0544,2025-11-23 19:21:16,True,EMP0710,2025-11-24 17:19:19,21.967500,Medium
1066,EVT0008490,PRJ0105,EMP0802,2025-02-20 08:56:26,True,EMP0942,2025-02-21 07:21:28,22.417222,Low
1483,EVT0015079,PRJ0322,EMP0268,2025-02-13 19:25:03,True,EMP0283,2025-02-14 19:25:57,24.015000,Medium


## daily_event_volume — running total by event_type

In [5]:
read_table("daily_event_volume").sort_values(["event_type", "event_date"]).head(15)

,event_type,event_count,cumulative_count,event_date
11,approval_granted,4,4,2025-01-01
24,approval_granted,8,12,2025-01-02
37,approval_granted,6,18,2025-01-03
53,approval_granted,4,22,2025-01-04
69,approval_granted,3,25,2025-01-05
82,approval_granted,2,27,2025-01-06
97,approval_granted,4,31,2025-01-07
109,approval_granted,5,36,2025-01-08
123,approval_granted,4,40,2025-01-09
135,approval_granted,8,48,2025-01-10


## peak_usage_analysis — top 20 date x hour

In [6]:
read_table("peak_usage_analysis", partitioned=False)

,event_date,event_hour,total_events,unique_users,event_types_per_hour
0,2025-09-16,1,25,24,9
1,2025-07-11,13,25,24,10
2,2025-08-02,14,24,23,9
3,2025-07-19,20,24,24,8
4,2025-02-08,2,23,23,6
5,2025-05-21,11,23,22,12
6,2025-02-24,23,23,23,10
7,2025-10-17,2,23,23,9
8,2025-02-02,10,23,22,9
9,2025-02-01,7,23,23,8


## Kafka run summary

In [7]:
with open(KAFKA_SUMMARY) as f:
    kafka_summary = json.load(f)

print(json.dumps(kafka_summary, indent=2))

{
  "run_at": "2026-09-24T13:46:46.159439+00:00",
  "topic": "presight.project.events",
  "total_messages_consumed": 8333,
  "event_counts": {
    "login": 2121,
    "comment_added": 325,
    "budget_updated": 252,
    "approval_granted": 136,
    "approval_rejected": 26,
    "project_status_change": 648,
    "document_uploaded": 524,
    "meeting_scheduled": 333,
    "task_completed": 849,
    "escalation_raised": 178,
    "approval_requested": 161,
    "task_created": 983,
    "logout": 1623,
    "escalation_resolved": 158,
    "milestone_reached": 16
  },
  "critical_escalations_forwarded": 14,
  "elapsed_seconds": 13.8,
  "throughput_messages_per_sec": 603.95
}


In [8]:
# Scratch cell — edit this query freely, e.g. any Spark table above
read_table("project_activity_summary").sort_values("escalation_count", ascending=False).head(10)

,project_id,total_events,escalation_count,task_completions,document_uploads,last_event_timestamp,unique_users,unique_event_types
217,PRJ0045,112,10,20,11,2025-12-30 01:43:58,108,11
62,PRJ0483,122,9,24,16,2025-12-19 04:04:47,114,11
24,PRJ0252,127,9,21,8,2025-12-23 20:47:58,113,12
433,PRJ0044,98,9,18,9,2025-12-24 12:09:58,94,11
52,PRJ0428,123,9,24,14,2025-12-26 08:04:38,116,13
466,PRJ0231,94,8,11,9,2025-12-30 03:55:21,87,11
298,PRJ0176,108,8,18,11,2025-12-24 20:42:13,105,12
8,PRJ0052,133,8,20,7,2025-12-29 18:37:17,127,12
261,PRJ0039,110,8,18,11,2025-12-23 01:29:18,106,12
37,PRJ0002,125,8,23,15,2025-12-27 22:20:26,122,12
